In [1]:
%load_ext memory_profiler

In [2]:
import os
import time
from memory_profiler import memory_usage
import subprocess
import pandas as pd

In [3]:
from agtools.core.unitig_graph import UnitigGraph

In [4]:
# Replace this with your actual function
def my_function(file_path):
    # Placeholder logic
    return UnitigGraph.from_gfa(file_path)

# Helper to run grep counts
def grep_count(char, file_path):
    try:
        out = subprocess.check_output(["grep", f"^{char}", file_path])
        return len(out.decode().splitlines())
    except subprocess.CalledProcessError:
        return 0

# Main profiling wrapper
def profile_function_on_files(files):
    results = []

    for file_path in files:
        print(f"Processing: {file_path}")

        # Time + memory usage
        start_time = time.time()

        mem_usage, result = memory_usage(
            (my_function, (file_path,)),
            retval=True,
            interval=0.1,
            max_iterations=1,
            include_children=True
        )
        elapsed_time = time.time() - start_time

        mem_increment = max(mem_usage) - min(mem_usage)  # increase over time
        max_mem = max(mem_usage)  # peak usage

        file_size = os.path.getsize(file_path) / (1024 * 1024)

        # GFA line stats
        count_S = grep_count("S", file_path)
        count_L = grep_count("L", file_path)
        count_P = grep_count("P", file_path)

        results.append({
            "file": file_path,
            "elapsed time (seconds)": round(elapsed_time, 4),
            "mem increment (MB)": round(mem_increment, 4),
            "max mem (MB)": round(max_mem, 4),
            "file size (MB)": file_size,
            "# S lines": count_S,
            "# L lines": count_L,
            "# P lines": count_P,
        })

    return pd.DataFrame(results)

In [5]:
gfa_files = [
    "/scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF23101184_AAF7F33M5/megahit/GSV327_AAF7F33M5_CAAGGATCGA-TTGTGTCTGC/output/final.gfa",
    "/scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF23101184_AAF7F33M5/megahit/GSV317_AAF7F33M5_GTATCTGAGG-CAAGATACGT/output/final.gfa",
    "/scratch/user/edwa0468/FAME/Projects/SAGCQA0428/PortJackson/megahit/SAGCFN_22_00686_S34/output/final.gfa",
    "/scratch/user/edwa0468/FAME/Projects/SAGCQA0545_combined/megahit/1834617_20180501_S_R1.fastq.gz/output/final.gfa",
    "/scratch/user/edwa0468/FAME/Projects/SAGCQA1236/megahit/24-05465_S6/output/final.gfa",
]

# Run profiling
df_results = profile_function_on_files(gfa_files)


Processing: /scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF23101184_AAF7F33M5/megahit/GSV327_AAF7F33M5_CAAGGATCGA-TTGTGTCTGC/output/final.gfa
Processing: /scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF23101184_AAF7F33M5/megahit/GSV317_AAF7F33M5_GTATCTGAGG-CAAGATACGT/output/final.gfa
Processing: /scratch/user/edwa0468/FAME/Projects/SAGCQA0428/PortJackson/megahit/SAGCFN_22_00686_S34/output/final.gfa
Processing: /scratch/user/edwa0468/FAME/Projects/SAGCQA0545_combined/megahit/1834617_20180501_S_R1.fastq.gz/output/final.gfa
Processing: /scratch/user/edwa0468/FAME/Projects/SAGCQA1236/megahit/24-05465_S6/output/final.gfa


In [6]:
# Show results
df_results

,file,elapsed time (seconds),mem increment (MB),max mem (MB),file size (MB),# S lines,# L lines,# P lines
0,/scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF2310...,3.1112,141.7734,264.9805,58.796576,123617,454,0
1,/scratch/user/edwa0468/Liz/Emma/AGRF_CAGRF2310...,2.4625,193.5039,401.0078,58.485081,94710,126,0
2,/scratch/user/edwa0468/FAME/Projects/SAGCQA042...,2.8598,200.8984,430.9453,58.348789,117839,3964,0
3,/scratch/user/edwa0468/FAME/Projects/SAGCQA054...,1.3667,196.7969,426.9297,58.287219,51873,142,0
4,/scratch/user/edwa0468/FAME/Projects/SAGCQA123...,2.9028,194.9492,423.2422,57.826014,107847,44,0
